In [ ]:
from pathlib import Path
import shutil 
import json

In [ ]:
dir_a = Path("home/a/projects/shroud/datasets/NST_JOJO/nst_jojo_new/")

In [12]:
filename = Path("/home/a/projects/shroud/datasets/NST_JOJO/nst_jojo_new/ffhq_rembg_rgb_jojo_enrie_00005.jpg")

stem = filename.stem.split("_")[-1]
print(stem)

filename = Path("/home/a/projects/shroud/datasets/ffhq/ffhq/00005.png")
stem = filename.stem.split("_")[-1]
print(stem)

00005
00005


In [ ]:
VALID_EXTENSIONS = {'.png', '.jpg', '.jpeg'}

DIR_NST = "/home/a/projects/shroud/datasets/NST_FFHQ_pair_dir/NST_JOJO"
DIR_FFHQ = "/home/a/projects/shroud/datasets/NST_FFHQ_pair_dir/ffhq"
DIR_UNPAIRED = "/home/a/projects/shroud/datasets/NST_FFHQ_pair_dir"




def get_file_stem(file_path):
    # get stem (filename without extension) and get last split 
    # i.e. ffhq_rembg_rgb_jojo_enrie_00005.jpg to 00005 or 00005.png to 00005
    return file_path.stem.split("_")[-1]

def get_stem_map(sub_dir):
    # for files in dir, if file and is image and add to set
    # Returns: {'00005': Path('/.../00005.png'), ...}
    return {get_file_stem(f): f for f in sub_dir.iterdir() if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS}


def process_paired_dir(dir_a,dir_b, dir_unpaired):
    manifest = []
    # get subdirectories
    subdirs_a = sorted([sub_dir for sub_dir in Path(dir_a).iterdir() if sub_dir.is_dir()])
    subdirs_b = sorted([sub_dir for sub_dir in Path(dir_b).iterdir() if sub_dir.is_dir()])
    # pair subdirectories
    for sub_a_paired, sub_b_paired in zip(subdirs_a, subdirs_b):
        print(f"Comparing: {sub_a_paired.name} <-> {sub_b_paired.name}")

        stem_map_a = get_stem_map(sub_a_paired)
        stem_map_b = get_stem_map(sub_b_paired) 

        #files in unprocessed missing in processed
        missing_files_in_b = set(stem_map_a.keys()) - set(stem_map_b.keys())
        missing_files_in_a = set(stem_map_b.keys()) - set(stem_map_a.keys())

        #copy unpaired to new dir
        for key in missing_files_in_b:
            src_path = stem_map_a[key]

            dest_dir = Path(dir_unpaired) / sub_a_paired.name
            dest_dir.mkdir(parents=True, exist_ok=True)

            print(f"Moving missing file: {src_path.name}")
            shutil.copy(stem_map_a[key], Path(dir_unpaired)/sub_a_paired.name/stem_map_a[key]) 

            manifest.append({
                "original_path": str(stem_map_a[key]),
                "copied to": str(dest_dir/sub_a_paired.name)
            })

        manifest.append({
            "---------------------------------------------------------------------"
        })

        for key in missing_files_in_a:
            src_path = stem_map_b[key]

            dest_dir = Path(dir_unpaired) / sub_b_paired.name
            dest_dir.mkdir(parents=True, exist_ok=True)

            print(f"Moving missing file: {src_path.name}")
            shutil.copy(stem_map_b[key], Path(dir_unpaired)/sub_b_paired.name/stem_map_b[key])

            manifest.append({
                "original_path": str(stem_map_b[key]),
                "copied to": str(dest_dir/sub_b_paired.name)
            })

    with open('unpaired_manifest.json', 'w') as f:
        json.dump(manifest, f, indent=4)
    print(f"Finished. Manifest saved with {len(manifest)} entries.")
        #remove unpaired from old dir


# process_paired_dir(DIR_FFHQ,DIR_NST, DIR_UNPAIRED)


In [ ]:
def cleanup_unpaired(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    
    print(f"Found {len(manifest)} files to clean up.")

    for entry in manifest:
        original = Path(entry['original_path'])

        if original.exists():
            original.unlink()
            print(f"Deleted: {original}")

cleanup_unpaired('unpaired_manifest.json')
    
    